# report01 — 통제 환경 — 반무향 챔버

**핵심.** 패시브 레이더로 **어떤 신호가 드론을 잘 비추나**를 공정히 재려면, 잡음·클러터가 매 순간 바뀌는 실환경이 아니라 **통제된 방**이 필요하다 — 30×20×11 m semi-anechoic(반무향) 챔버. 그리고 이 방의 환경 전파는 **Sionna RT 가 계산한다** — 그 계산이 우리 씬 위에서 닫힌형 물리(거울상+프레넬)와 자기일치함을 §4 에서 확인한다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | 실환경은 지나가는 차·옆 건물·날씨·간섭으로 매 순간 바뀌어, 신호를 바꿔가며 재는 **공정 비교가 불가능**하다. 이 통제(반사·잡음 제거)는 스톡 시뮬이 자동으로 주지 않는다 — 반사면을 하나(바닥)로 줄인 반무향 챔버를 형상·재질까지 설계·모델링해야 얻어진다(바닥 콘크리트·차폐 금속은 ITU-R P.2040, 흡수체 폼은 ITU 에 없어 모델값 — §1·§2). |
| **② 선행 연구의 방식** | 최신 ISAC 연구들은 멀티패스·지연·도플러 **채널을 Sionna RT 로 계산**한다 — Deterministic-Modeling(arXiv:2603.28736, EuCAP 2026) · LAMBDA(arXiv:2607.03826) · Temporal-GNN(arXiv:2604.08306) · CISSIR(arXiv:2502.10371, NVIDIA 공식 'Made with Sionna'). 환경 전파를 RT 에 맡기는 것은 우리만의 선택이 아니다(§3). |
| **③ 쓴 라이브러리·결합** | **Sionna RT `PathSolver`/`RadioMapSolver` + `Scene.render()`**(Mitsuba 3 / OptiX, GPU). 씬을 주면 경로별 지연 τ·복소이득·반사점·도플러를 광선추적으로 반환한다 — 챔버 형상·재질만 우리가 얹고 전파 계산 엔진은 라이브러리 그대로 재사용한다(§4). |
| **④ 검증** | 라이브러리를 그냥 믿지 않고 **바닥 1회 반사**(설계상 가장 강하게 남긴 반사면)의 지연·감쇠를 **image-source + 프레넬** 손계산과 대조 → 지연 차 5.8e-06 ns·세기 차 **4.5e-04 dB**. ⚠ Sionna 도 같은 image method·같은 프레넬 식을 쓰므로 이 일치는 **독립 검증이 아니라 구현 일관성**(씬 좌표·단위·프리미티브 식별) 확인이다(§4). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 챔버 형상·치수 | src/chamber.py (30×20×11 m, EMC 반무향 표준형) | 설계 |
| 재질 전기상수 | ITU-R P.2040 (콘크리트·금속) + 흡수체 폼 | 표준 |
| 바닥 반사 손계산 | 거울상(image source) + 프레넬 반사계수 | 물리 |
| 전파 경로·렌더 | Sionna RT (Mitsuba 3 / OptiX) | 시뮬 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-rt` | Sionna RT `PathSolver` — 전파 광선추적. 경로별 **지연 τ · 도플러 f_d · 복소이득 · 반사점 좌표**를 준다 | 🟢 **Sionna 내부** (Mitsuba 3 / OptiX, GPU) |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `sionna-radiomap` | Sionna RT `RadioMapSolver` — 공간별 전파 세기 분포 | 🟢 **Sionna 내부** (GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 렌더·라디오맵·경로 검증 모두 GPU 1장에서 수 분. 무거운 계산 아님.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 챔버 씬 렌더 + 라디오맵 (Sionna RT)
python src/render_rt.py
# 바닥 반사: 손계산 vs Sionna RT 검증 → outputs/report3_rt.json
python benchmark/rt_experiments.py   # S2_floor 블록 생성
# 이 노트북 재생성
python src/make_notebook01.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report1.json` | 챔버 치수·재질(단일 진리원) |
| `outputs/report3_rt.json` | 바닥 반사 손계산 vs Sionna RT (레거시 report3_ 파일명 유지) |
| `outputs/renders/r1_1x_chamber_*.png` | Sionna 가 렌더한 챔버 사진 |
| `outputs/figures/report3_f4_floor.png` | 바닥 반사 일치 그래프 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- 이 리포트는 **방(환경) 그 자체**만 다룬다. 방에 띄울 드론은 report02, 드론이 되쏘는 밝기(RCS)는 report06.
- Sionna 기본 solver 는 표면에서의 **정반사(specular) 전파**를 계산한다 — 표적을 광선으로 조준해 그 밝기(σ)를 뽑는 **산란적분 단계는 들어 있지 않다.** 표적 밝기는 별도 SBR 로 구하며 report06 에서 다룬다.
- 바닥 반사가 탐지 결과(오검출)에 어떻게 번지는지는 여기서 다루지 않는다 → report09.
- §4 의 손계산 대조는 **구현 일관성 확인**이지 전파 물리의 독립 검증이 아니다 — Sionna 의 정반사 경로탐색도 **image method**, 반사계수도 **같은 프레넬 식**이라 지연·세기 둘 다 같은 알고리즘의 자기일치다. 실측·풀웨이브 EM 과의 대조는 이 리포트에 없다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| (이 리포트가 첫 편) | 전체 시리즈의 무대를 세운다 |
| → report02 | 이 방에 띄울 드론을 만든다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **패시브 레이더** | 자기 송신기 없이 **주변에 이미 있는 전파**(WiFi·방송·이동통신)를 빌려 표적을 탐지하는 레이더 |
| **semi-anechoic(반무향)** | 벽·천장은 전파를 크게 흡수하고 **바닥을 주 반사면으로 남긴** 방. 무향실 + 반사 바닥 |
| **흡수체(absorber)** | 전파를 되튕기지 않고 열로 먹어 없애는 피라미드 모양의 폼. 벽·천장에 붙인다 |
| **정반사(specular)** | 거울처럼 '들어온 각 = 나가는 각'으로 딱 한 방향으로 튕기는 반사 |
| **거울상(image source)** | 바닥을 거울로 보고 수신기를 바닥 아래로 대칭 복사해 반사 경로를 직선처럼 푸는 계산법 |
| **프레넬 반사계수 Γ** | 표면이 전파를 얼마나 되튕기는지(0~1). 각도·재질·편파로 정해진다 |
| **dB** | 세기의 로그 단위. −3 dB=절반, −10 dB=1/10, -14.7 dB≈1/29 세기 |
| **Sionna RT** | 전파를 광선으로 추적해 경로별 지연·세기·도플러를 주는 오픈소스 레이트레이서(NVIDIA) |

</details>

---


---
## §1. 왜 하필 '통제된 방' 안인가

### 이 레이더는 자기 전파를 안 쏜다

보통 레이더는 자기가 강한 전파를 '쾅' 쏘고, 표적에 맞고 돌아오는 메아리를 듣는다. 이 실험이 쓰는 건 **패시브 레이더(자기 송신기 없이 주변 전파를 빌려 쓰는 레이더)** 다. 방 안에는 이미 **WiFi 공유기, 휴대폰 기지국(LTE), 5G** 가 늘 전파를 뿌리고 있다. 패시브 레이더는 이 **남의 전파**가 드론에 맞고 되돌아오는 아주 약한 메아리를 엿들어 드론을 찾는다.

### 빌린 신호끼리 공정하게 겨루려면

빌릴 수 있는 신호는 여러 종류다 — WiFi 는 대역폭이 넓고, LTE 는 멀리 가고, 5G 는 촘촘하다. **어떤 신호가 드론을 제일 잘 비추나?** 이걸 제대로 비교하려면 신호 말고 **나머지 조건은 전부 똑같이 고정**해야 한다. 실환경에서는 그게 불가능하다. 지나가는 차, 옆 건물 벽, 날씨, 다른 기지국의 간섭 — 매 순간 바뀌는 잡음과 반사가 결과를 오염시켜, 결과의 차이가 신호 때문인지 환경 때문인지 가를 수 없다. 게다가 이 통제(반사·잡음 제거)는 시뮬레이터를 켠다고 저절로 생기지 않는다 — 반사면을 실제로 없앤 방을 형상·재질까지 지어 넣어야 얻어진다.

그래서 실험 전체를 **통제된 방** 안에 넣는다. 이 방은 반사면을 딱 하나(바닥)만 남기도록 설계한 방이다. 그러면 신호를 바꿔가며 실험할 때 **결과의 차이가 오로지 신호 때문**이라고 말할 수 있다. 그리고 이 방 안의 전파는 **Sionna RT 라이브러리가 계산한다**(§4 에서 닫힌형 손계산과 대조한다). 다음 절에서 그 방이 실제로 어떻게 생겼는지 본다.

---
## §2. 그 방 — 30×20×11 m 반무향 챔버

방의 크기는 가로 **30 m** × 세로 **20 m** × 높이 **11 m** 다. 농구 코트 두 개를 이어 붙인 정도의, 드론 몇 대가 넉넉히 날 수 있는 큰 실내다.

### 다섯 면은 흡수, 한 면은 반사

이 방의 핵심 성질은 **semi-anechoic(반무향)** 이라는 것이다. 말을 풀면 이렇다:

- **네 벽 + 천장** = **피라미드 흡수체**로 덮여 있다. 전파가 여기 닿으면 대부분 폼 속에서 열로 사라진다(설계목표 −25 dB). 단 **완전히 없애지는 못한다** — 이 씬의 단일면 실효 |Γ|≈0.55 라 흡수체도 되쏜다(아래 ⚠). (모델 안에서 흡수체 표면은 삼각형 약 18,392개로 촘촘히 만들어져 있다.)
- **바닥** = 콘크리트다. **설계상 가장 강하게 되튕기도록 남겨둔** 면이다. 콘크리트의 유전율은 ε_r=5.24, 반사율은 |Γ|≈0.39(수직 입사 기준 — 실제 46° 입사에선 §4 의 Γ≈0.26 — **TM(V, 입사면 평행) 기준**이며 TE 는 0.52 로 오히려 커진다) — 닿은 전파의 **진폭** 기준 약 39%(세기로는 약 16%)를 되튕긴다.

왜 바닥은 안 덮나? 실제 EMC(전자파) 시험실이 원래 이런 구조다 — 장비를 바닥에 놓아야 하니 바닥은 흡수체를 못 깐다. 이 실험의 챔버는 그 **표준형 시험실**을 그대로 본떴다. 덕분에 **설계상 남겨둔 반사면이 하나(바닥)** 로 줄어든다 — 흡수체 잔여 반사가 0 은 아니라는 점은 아래 ⚠ 에서 수치로 짚는다. 반사면이 하나로 줄면 그 하나가 맞는지 손으로 검산할 수 있고(§4), 나중에 신호끼리 비교할 때도 조건이 단순명료해진다.

### Sionna 가 렌더한 실제 방

아래는 이 방을 **Sionna 의 렌더러가 사진처럼 그려낸** 모습이다. 벽·천장을 뒤덮은 뾰족한 피라미드 폼(흡수체)과, 반짝이는 체크무늬 바닥(반사하는 콘크리트)이 한눈에 보인다. 빨간 점이 송신기(TX, 빌려 쓰는 전파의 출처), 초록 점이 수신기(RX, 메아리를 듣는 귀), 바닥 가운데 작은 물체가 드론이다.

![Sionna 가 렌더한 챔버 내부 — 벽·천장은 흡수체 폼, 바닥은 반사 콘크리트](outputs/renders/r1_11_chamber_wide.png)

*벽·천장의 피라미드 흡수체가 전파를 먹어 없앤다. 설계상 반사면으로 남긴 것은 바닥뿐이다(흡수체 잔여 반사는 0 이 아니다 — 아래 ⚠). 빨강=송신기, 초록=수신기.*

같은 방을 위에서 내려다본 그림과, 바닥을 스치듯 낮게 본 그림이다.

![위에서 본 챔버](outputs/renders/r1_12_chamber_top.png)

![바닥을 스치듯 본 챔버 — 반사면인 바닥이 강조된다](outputs/renders/r1_13_chamber_grazing.png)

### 방의 도면과 전파가 흐르는 길

다음은 같은 방을 **평면도(위에서)와 단면도(옆에서)** 로 그린 것이다. 송신기 TX 와 수신기 RX 는 바닥에서 떨어진 곳에 두었고, 둘 사이 직선거리(기준선)는 약 **15 m** 다. 단면도를 보면 전파가 표적을 지나 곧장 오는 길(점선)과, **바닥에 한 번 튕겨서 오는 길**(주황 실선)이 함께 보인다. 다섯 면이 전파를 크게 줄이니, 바닥은 **설계상 가장 강한 반사면**이다. ⚠ 다만 '방 안 유일한 반사'는 아니다 — 우리 RT 해가 그걸 반증한다: 저심도 정반사에서 **천장 흡수체(`absorber_ceiling`) 1회 반사가 -9.8 dB(13.0 ns)로 바닥(-14.7 dB)보다 오히려 세고**, 정면 흡수체 2회 반사(-11.6 dB)는 바닥과 같은 지연 빈(19.3 ns)에 3.1 dB 더 세게 들어온다. 즉 챔버의 실제 통제는 '물리적 단일반사'가 아니라 **정적 클러터를 ECA 가 도플러 0 에서 제거**하는 데서 온다(report09).

![챔버 도면 — 평면도와 단면도, 흡수 다섯 면과 반사 바닥](outputs/figures/report1_chamber_geometry.png)

*평면도(왼쪽)·단면도(오른쪽). 벽·천장은 흡수, 바닥은 콘크리트 반사. 주황 선이 바닥에서 튕겨 오는 경로다.*

반사면이 이 하나로 줄면, 방의 전파를 **어느 도구로 계산하고 어떻게 대조할지**가 단순해진다. 먼저 선행 연구가 환경 전파를 어떻게 다루는지 보고(§3), 그다음 우리가 그걸 Sionna RT 로 계산해 이 바닥 반사 하나로 닫힌형 물리와 맞댄다(§4).

![.](outputs/renders/anim/orbit_chamber.gif)

<sub>챔버를 한 바퀴 도는 Sionna RT 렌더 — 송신기(빨강)·수신기(초록)·바닥 반사·표적 배치.</sub>

---
## §3. 선행 연구는 환경 전파를 어떻게 계산하나

방 안의 전파 — 곧장 오는 신호, 벽에 흡수되는 신호, 바닥에 튕기는 신호, 각각의 도착 시간·세기·도플러 — 를 무엇으로 계산할까. 환경 전파(멀티패스·지연·도플러 채널)를 **Sionna RT 로 계산**하는 것은 우리만의 선택이 아니라 최신 ISAC 연구가 공통으로 택한 방식이다:

| 선행 연구 | 환경 전파를 어떻게 |
|---|---|
| Deterministic-Modeling (arXiv:2603.28736, EuCAP 2026) | 동적 ISAC 채널을 Sionna RT 로 결정론적 생성 |
| LAMBDA (arXiv:2607.03826) | UAV 센싱 데이터셋의 멀티패스 채널을 Sionna RT 로 |
| Temporal-GNN (arXiv:2604.08306) | ISAC 표적 검출·추적의 지연·도플러 채널을 Sionna RT 로 |
| CISSIR (arXiv:2502.10371) | **NVIDIA 공식 'Made with Sionna'** ISAC 등재 — Sionna RT 로 빔·자기간섭 |

공통점은 **표적 밝기(RCS)를 뺀 전파 경로 계산은 Sionna RT 에 맡긴다**는 것이다 — 정반사·지연·도플러는 산란적분이 필요 없어(거울반사만으로 물리적으로 옳음) RT 가 잘하는 부분이기 때문이다. 우리도 같은 판단으로 이 방의 환경 전파를 Sionna RT 로 계산하되, 라이브러리를 그냥 믿지 않고 이 방에서 설계상 가장 강하게 남긴 반사(바닥)로 **닫힌형 물리 대조**를 붙인다 — 그 대조가 무엇까지 보증하는지도 같은 절에서 못박는다(§4).

---
## §4. 우리가 쓴 방식 — Sionna RT 로 계산하고 바닥 반사로 검증

이 방 안의 전파를 **Sionna RT** 로 계산한다. 씬을 주면 전파를 광선으로 추적해 경로별 **지연 · 복소이득 · 반사점 · 도플러**를 돌려주는 오픈소스 라이브러리다 — `PathSolver`·`RadioMapSolver`·`Scene.render()`(Mitsuba 3 / OptiX, GPU). 챔버 형상·재질(ITU-R P.2040)만 우리가 얹고 전파 계산 엔진은 **라이브러리 그대로** 재사용한다.

라이브러리를 그냥 믿지 않고 **닫힌형 물리로 맞댄다**. 대조할 대상은 이 방에서 설계상 가장 강하게 남긴 반사 — **바닥에서 한 번 튕겨 오는 전파** 다. 이 하나를 교과서 물리로 손계산해 Sionna RT 의 답과 나란히 놓는다 (무엇이 검증되고 무엇이 안 되는지는 표 아래에서 정확히 긋는다).

### 교과서 물리로 손계산 — 거울상 + 프레넬

전파가 바닥에 튕겨 수신기로 가는 길은 중학교 수준의 거울 문제로 바꿀 수 있다.

> 🪞 **비유.** 잔잔한 호숫가에서 맞은편 사람을 볼 때, 물에 비친 그 사람의 **거울상**을 향해 직선을 그으면 빛이 물에 튕겨 오는 길이 그대로 나온다. 바닥을 호수 삼아 수신기를 바닥 아래로 뒤집어 복사하면(=거울상), 튕겨 오는 경로가 **직선 하나**로 펴진다.

이 거울상 방법으로 두 가지를 닫힌형으로 구한다:

1. **얼마나 늦게 오나 (지연).** 곧장 오는 길은 약 15.1 m, 바닥에 튕겨 오는 길은 약 20.9 m 다. 차이가 5.8 m 이니, 빛의 속도로 이 거리만큼 더 도는 데 걸리는 시간은 **19.3 ns**(나노초). 즉 바닥 메아리는 직접파보다 그만큼 늦게 도착한다.
2. **얼마나 약해지나 (세기).** 두 가지가 세기를 깎는다. ① 더 멀리 돌아오니 퍼져서 약해지고(약 2.8 dB), ② 콘크리트 바닥이 완벽한 거울이 아니라 일부만 되튕긴다. 바닥에 닿는 각도(46° — 수직 기준)와 콘크리트 물성으로 정해지는 **프레넬 반사계수 Γ≈0.26** 가 그 감쇠를 준다. 둘을 합치면 바닥 메아리는 직접파보다 **14.7 dB**(진폭으로 약 5배·세기로는 약 29배) 약하다.

### 라이브러리의 답과 맞댄다

이제 같은 방을 **Sionna RT** 에게 주고 전파를 광선으로 추적하게 한다. Sionna 는 우리 손계산을 참고하지 않고 씬 기하만으로, 광선이 바닥(`floor_light`)에 **한 번(1회) 튕겨** 수신기로 오는 경로를 찾아낸다. 두 답을 나란히 놓으면:

| 무엇을 | 닫힌형 물리 (거울상+프레넬) | Sionna RT (라이브러리) | 차이 |
|---|---|---|---|
| 늦게 오는 시간 | 19.31 ns | 19.31 ns | 5.85e-06 ns |
| 약해진 세기 | -14.68 dB | -14.68 dB | 4.53e-04 dB |

**시간도 세기도 사실상 완전히 겹친다** — 세기 차이 4.5e-04 dB, 지연 차이 5.8e-06 ns (반올림해 '0.00' 으로 보이지 않도록 지수로 적는다).

> ⚠ **이 일치는 독립 검증이 아니다.** 두 축 모두 *같은 알고리즘끼리의 자기일치*다.
> - **세기** — Sionna 의 정반사 감쇠도, 우리 손계산도 **같은 프레넬 식·같은 ε_r** 을 쓴다(Sionna RT 기술보고서 arXiv:2504.21719 의 프레넬 식 (127)·(128) ↔ 우리 `benchmark/geometry.py:92-100` `_fresnel_floor`). 세기가 맞는 건 사실상 항등식이다.
> - **지연** — Sionna 의 정반사 경로탐색 **자체가 image method** 다(설치본 `sionna/rt/path_solvers/image_method.py`; 같은 기술보고서 §1 *"Sionna RT integrates shooting and bouncing of rays (SBR) with the image method"*, §3.2 *Image Method-based Candidate Processing*). 우리 손계산도 바닥에 RX 를 거울복사해 편 것이므로, 기하 역시 같은 구성이다 — 지연도 독립 증거가 아니다.
>
> 그래서 이 대조가 실제로 보증하는 것은 **씬 좌표·단위·프리미티브 식별의 구현 일관성** 이다 — 챔버 기하가 의도한 좌표에 서 있고, 바닥면이 z=0 에 있고, 반사점(`floor_light`)이 옳게 식별되며, 길이·시간·dB 규약이 두 계산 사이에서 어긋나지 않는다는 확인. **전파 물리의 독립 검증(실측 또는 풀웨이브 EM 대조)은 이 리포트에 없다.**

![바닥 반사: 손계산 vs Sionna — 지연·세기 모두 일치](outputs/figures/report3_f4_floor.png)

*왼쪽: 거울상으로 편 반사 경로(직접 15.07 m vs 바닥 경유 20.86 m). 가운데·오른쪽: 지연(19.3 ns)과 세기(14.7 dB 약함)가 닫힌형 물리와 Sionna RT 에서 겹친다.*

### Sionna 가 추적한 광선 그림

Sionna RT 가 실제로 찾아낸 광선들을 방 안에 그려 보면 이렇다. 송신기에서 나온 전파가 표적을 지나 수신기로 가는 길, 그리고 **바닥에 한 번 튕겨 가는 길**이 함께 보인다.

![Sionna 가 추적한 1회 반사 경로 (비스듬히)](outputs/renders/rt_10_paths_1bounce.png)

![같은 경로 — 위에서 / 옆에서](outputs/renders/rt_12_paths_1bounce_side.png)

### 왜 환경 전파를 Sionna 에 맡기나

방 안의 전파가 흐르는 방식 — 직접 오는 신호, 벽에 흡수되는 신호, 바닥에 튕기는 신호, 각각의 도착 시간과 세기 — 이 **환경 전파**는 Sionna RT 가 하도록 설계된 바로 그 일이고, 최신 ISAC 연구들이 공통으로 맡기는 부분이다(§3). 위 대조는 그 계산이 **우리 씬 위에서 닫힌형 물리와 자기일치**함을 보인다(독립 검증은 아니다 — 위 ⚠). 그래서 앞으로 이 시리즈의 모든 실험은 **이 방을, 이 라이브러리 계산 위에** 올려놓는다.

> 📌 짚어둘 경계. Sionna RT 는 전파가 표면에서 **어떻게 반사·전달되는지(환경)** 를 계산한다. 하지만 표적을 광선으로 조준해 그 **되비침 밝기(RCS, 표적이 얼마나 밝게 되쏘나)** 를 뽑는 산란적분 단계는 기본 solver 에 없다 — 선행 연구도 같은 경계를 지적한다. 그 밝기는 별도 방법(SBR+PO)으로 구하며 **report06** 에서 다룬다. 여기서 중요한 건 하나다 — **방(환경)의 전파는 Sionna RT 가 믿을 만하게 계산한다.**

![챔버 내부의 전력장 — Sionna RT 가 계산한 결과](outputs/renders/rt_20_radiomap_droneplane.png)

<sub>이것이 Sionna RT 가 실제로 내놓는 것이다 — 송신점(빨강)에서 나간 전파가 챔버 안 각 지점에 만드는 **수신 전력장**(바닥 히트맵). 흡수체 벽 쪽은 어둡고(반사 약함) 반사면 바닥을 따라 밝은 무늬가 퍼진다. 표적(드론) 하나가 이 장 안에 놓여 있다. 방의 전파 분포는 이렇게 신뢰성 있게 계산되지만, 그 표적이 얼마나 밝게 **되쏘는지**(RCS)는 별도로 구한다 → **report06**.</sub>

---
## 정리

- 패시브 레이더는 **남의 전파(WiFi·LTE·5G)를 빌려** 드론을 본다. 무엇이 잘 보이는지 공정히 재려면 통제된 방이 필요하다.
- 그 방은 **30×20×11 m 반무향 챔버** — 벽·천장은 전파를 흡수하고 **바닥을 주 반사면으로 남긴다**. 덕분에 설계상 남겨둔 반사면이 하나로 줄어든다(흡수체 잔여 반사는 0 이 아니다 — 저심도에서 천장 -9.8 dB > 바닥 -14.7 dB, §2).
- 이 방의 환경 전파는 **Sionna RT 라이브러리**가 계산한다 — 여러 최근 ISAC 연구가 채널을 만드는 바로 그 도구다. 바닥 반사(19.3 ns 늦게, 14.7 dB 약하게)가 교과서 물리(거울상+프레넬)와 **4.5e-04 dB** 로 겹친다 — 단 둘이 같은 image method·같은 프레넬 식을 쓰므로 이것은 **구현 일관성 확인**이지 독립 검증이 아니다(§4).

이제 무대가 준비됐다. **다음 리포트(02): 이 방에 띄울 드론을 만든다** — 실제 DJI 기체를 본뜬 3D 모델을 세우고, 그 모양이 스펙과 맞는지 검증한다.